# arXiv 2010.07404 — Full paper grid on a free Colab T4 GPU

Runs the full 11-month train / 3-month test BTC pipeline plus ETH/BCH/LTC/EOS transfer on a Colab T4. End-to-end wall time on a T4 is roughly 4–8 hours; on a free Colab session you may need to split the run across two sessions.

**Before you start**

1. `Runtime` → `Change runtime type` → **GPU (T4)**.
2. Run the cells in order. The first cell mounts Google Drive so the fetched 5–15 GB of trade data and the trained models survive across Colab disconnects.

## 1. Mount Google Drive (cache the dataset across sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/lstm_2010_07404')
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
print('Working dir:', os.getcwd())

## 2. Clone the repo (or pull updates if it already exists)

In [ ]:
REPO = 'A-Deep-Learning-Framework-for-Predicting-Digital-Asset-Price-Movement-from-Trade-by-trade-Data'
if not pathlib.Path(REPO).exists():
    !git clone https://github.com/khangnguyen2705/{REPO}.git
else:
    !cd {REPO} && git pull
os.chdir(REPO)
print('Repo dir:', os.getcwd())

## 3. Install dependencies (~2 min)

In [ ]:
!pip install -q -r requirements.txt

## 4. Drop the macOS-only TF eager-mode patch

The patch in `run.py` works around an Apple Silicon LSTM hang. On a Linux T4 it just slows things down ~50×, so we strip it out.

In [ ]:
import re
p = pathlib.Path('run.py')
txt = p.read_text()
txt = re.sub(r'_tf\.config\.run_functions_eagerly\(True\)', '# (eager-mode patch disabled on Linux GPU)', txt)
p.write_text(txt)
print('Patched run.py')

## 5. Verify the GPU is visible to TensorFlow

In [ ]:
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

## 6. Fetch 11 months of BTC + 3 months of ETH/BCH/LTC/EOS aggTrades

Total ~5–15 GB. The fetcher caches each month under `data/raw/`; rerunning is idempotent. On a Colab session a fresh fetch takes 30–90 minutes depending on Binance throttling.

In [ ]:
!python -m src.data.fetch_binance --data-dir data 2>&1 | tail -50

## 7. Run the full paper grid

`max_epochs: 200`, `early_stop_patience: 20`, full `4 × 4 × 4` grid per setup. On a T4 the dominant cost is the longest sequences (`T=2000`); plan on 4–8 hours of GPU time. Output streams to `reports/run_paper.log`.

In [ ]:
!mkdir -p reports && PYTHONUNBUFFERED=1 python run.py --config configs/paper_2010_07404.yaml 2>&1 | tee reports/run_paper.log | grep -E '^(STAGE|SETUP|---|.{2,4}T= +[0-9]|.{2,4}.Best:|.{2,4}OOS|Trades:|Total return|Sharpe|Max drawdown|Win rate|h=|PIPELINE)'

## 8. Inspect the headline numbers

In [ ]:
import json, pandas as pd
metrics = json.load(open('reports/btc_out_of_sample_metrics.json'))
rows = []
for setup_horizon, payload in metrics.items():
    if setup_horizon == 'transfer': continue
    rows.append({
        'setup':        payload['setup'],
        'm':            payload['horizon_m'],
        'best_T':       payload['best_T'],
        'best_N':       payload['best_N'],
        'val_loss':     payload['val_loss'],
        'oos_acc':      payload['oos_accuracy'],
        'oos_loss':     payload['oos_loss'],
        'IC':           payload['ic']['ic'],
        'p_value':      payload['significance']['p_value'],
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print('Transfer:')
print(pd.read_csv('reports/transfer_other_pairs_accuracy.csv'))

## 9. Snapshot reports back to Drive

Colab disposes of `/content` between sessions. The cell below copies the whole `reports/` tree to your Drive so you can inspect the JSON / PNGs after disconnect.

In [ ]:
import shutil, datetime
stamp = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
dest = pathlib.Path(f'/content/drive/MyDrive/lstm_2010_07404/results_{stamp}')
shutil.copytree('reports', dest)
print('Copied to', dest)